# All candidate fish at shifted video frames, 3-fish recording

Only fish 0 emits, so every EOD should be picked as fish 0. My pick is the fish whose model (BEM σ, sink at the tail tip, the
other fish as bodies) is closest to the recorded pattern (L2 distance d, patterns × −1, max |v| = 1).

The full offset sweep (`EOD_offset_sweep_all_3fish.ipynb`) found that the model of fish 0 fits best with the masks 4 video frames
(80 ms) *before* each EOD's frame. Here all three fish are solved at −5, −4 and −3 frames, and the pick at each offset is compared with
the pick at the EOD's own frame (offset 0, the cached model).

- Fish 0 at every offset: `offset_sweep_all_good.csv` (`offset_sweep_runner.py`).
- Fish 1 and 2 at −5, −4, −3: `offset_all_fish_good.csv` (`offset_all_fish_runner.py`; offset 0 checked against the cache).
- Fish 1 and 2 at 0: the cached model.
- Denis' pick stays at the EOD's own frame (his pipeline was not rerun).

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import eod_examples as ex

RES = 'results/eod_ground_truth_3fish/conductivities'
OFFS = [0, -3, -4, -5]; FISH = [0, 1, 2]; KB = -4
c = ex.use_better_list(ex.load_recording('3-fish')); a = c['assigned']; PIDS = c['pair_ids']
S0 = pd.read_csv(f'{RES}/offset_sweep_all_good.csv').set_index('eod_row')
SO = pd.read_csv(f'{RES}/offset_all_fish_good.csv').set_index('eod_row')
idx = a.index[a['good frames'].to_numpy()]
assert set(idx) == set(S0.index) == set(SO.index), 'runs incomplete'
S0, SO = S0.loc[idx], SO.loc[idx]

BEM = 'BEM σ (interior 100, skin 0.05, 0.005 cm), sink at tail tip'
D = {}                                                    # offset -> (n_EOD, 3) distance of each fish
for o in OFFS:
    D[o] = np.column_stack([S0[f'd_{o}'].to_numpy()] + [(a.loc[idx, f'distance fish {f} | {BEM}'] if o == 0 else SO[f'd{f}_{o}']).to_numpy(float)
                                                       for f in (1, 2)])
def picks(Dm):
    ok = np.isfinite(Dm).any(axis=1); p = np.full(len(Dm), -1); p[ok] = np.nanargmin(Dm[ok], axis=1); return p
PK = {o: picks(D[o]) for o in OFFS}
den = a.loc[idx, "fish | Denis' pipeline (latest run)"].to_numpy()
print(f'{len(idx)} good-frame EODs')
print(f'pick at 0 equals the cached assignment in {(PK[0] == a.loc[idx, f"fish | {BEM}"].to_numpy()).mean() * 100:.2f} % of EODs')
for o in OFFS:
    print(f'  offset {o:+d}: EODs with no model for fish 0 / 1 / 2: ' + ' / '.join(str(int(np.isnan(D[o][:, f]).sum())) for f in FISH)
          + f'; no pick: {(PK[o] < 0).sum()}')

6751 good-frame EODs
pick at 0 equals the cached assignment in 100.00 % of EODs
  offset +0: EODs with no model for fish 0 / 1 / 2: 0 / 0 / 0; no pick: 0
  offset -3: EODs with no model for fish 0 / 1 / 2: 0 / 0 / 0; no pick: 0
  offset -4: EODs with no model for fish 0 / 1 / 2: 0 / 0 / 0; no pick: 0
  offset -5: EODs with no model for fish 0 / 1 / 2: 0 / 0 / 0; no pick: 0


## % correct (picked fish 0) by offset

In [2]:
n = len(idx)
rows = []
for o in OFFS:
    p = PK[o]; rows.append(dict(offset=o, ms=o * 20, pct_correct=(p == 0).mean() * 100, n_correct=(p == 0).sum(),
                                pct_fish1=(p == 1).mean() * 100, pct_fish2=(p == 2).mean() * 100, no_pick=(p < 0).sum(),
                                median_d0=np.nanmedian(D[o][:, 0]),
                                median_margin=np.nanmedian(np.nanmin(D[o][:, 1:], axis=1) - D[o][:, 0])))
T = pd.DataFrame(rows); print(T.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print(f"\nDenis (EOD's own frame): {(den == 0).mean() * 100:.2f} % correct")
x0, xb = PK[0] == 0, PK[KB] == 0
print(f'\noffset 0 -> {KB}: right -> right {(x0 & xb).sum()}, wrong -> right {(~x0 & xb).sum()}, right -> wrong {(x0 & ~xb).sum()}, '
      f'wrong -> wrong {(~x0 & ~xb).sum()}')
# paired test: McNemar (exact binomial on the discordant EODs)
from scipy.stats import binomtest
b, cc = (~x0 & xb).sum(), (x0 & ~xb).sum()
print(f'McNemar exact p = {binomtest(b, b + cc).pvalue:.1e}')
print('\npick at 0 (rows) vs pick at -4 (columns):'); print(pd.crosstab(pd.Series(PK[0], name='0'), pd.Series(PK[KB], name=str(KB))))

 offset   ms  pct_correct  n_correct  pct_fish1  pct_fish2  no_pick  median_d0  median_margin
      0    0       94.505       6380      2.207      3.288        0      0.560          1.114
     -3  -60       98.000       6616      0.622      1.378        0      0.391          1.262
     -4  -80       98.430       6645      0.444      1.126        0      0.364          1.291
     -5 -100       98.400       6643      0.415      1.185        0      0.376          1.286

Denis (EOD's own frame): 94.07 % correct

offset 0 -> -4: right -> right 6348, wrong -> right 297, right -> wrong 32, wrong -> wrong 74
McNemar exact p = 5.8e-55

pick at 0 (rows) vs pick at -4 (columns):
-4     0   1   2
0               
0   6348   6  26
1    123  24   2
2    174   0  48


## Where the shift helps: movement, recording quarter, against Denis

In [3]:
mv = np.hypot(S0[f'head_x_{KB}'] - S0['head_x_0'], S0[f'head_y_{KB}'] - S0['head_y_0']).to_numpy() / ex.PX_PER_CM
for lab, m in ((f'fish 0 head moved > 1 cm in {-KB * 20} ms', mv > 1), ('moved <= 1 cm', mv <= 1)):
    print(f'{lab}: {m.sum()} EODs, % correct {x0[m].mean() * 100:.1f} -> {xb[m].mean() * 100:.1f}')
fr = S0.frame.to_numpy(); q = pd.qcut(fr, 4, labels=False)
print()
for k in range(4):
    m = q == k
    print(f'{k + 1}. quarter (frames {fr[m].min()}–{fr[m].max()}): % correct {x0[m].mean() * 100:.1f} -> {xb[m].mean() * 100:.1f} '
          f'(Denis {(den[m] == 0).mean() * 100:.1f})')
dn = den == 0
print(f'\nagainst Denis (his own frame): at 0 both right {(x0 & dn).sum()}, only mine {(x0 & ~dn).sum()}, only Denis {(~x0 & dn).sum()}, '
      f'both wrong {(~x0 & ~dn).sum()}')
print(f'               at {KB}: both right {(xb & dn).sum()}, only mine {(xb & ~dn).sum()}, only Denis {(~xb & dn).sum()}, '
      f'both wrong {(~xb & ~dn).sum()}')

fish 0 head moved > 1 cm in 80 ms: 1762 EODs, % correct 89.0 -> 98.9
moved <= 1 cm: 4989 EODs, % correct 96.5 -> 98.3

1. quarter (frames 12–10741): % correct 93.1 -> 98.0 (Denis 93.7)
2. quarter (frames 10742–14591): % correct 93.0 -> 96.8 (Denis 93.0)
3. quarter (frames 14593–20174): % correct 96.1 -> 99.8 (Denis 95.0)
4. quarter (frames 20187–34094): % correct 95.9 -> 99.2 (Denis 94.7)

against Denis (his own frame): at 0 both right 6261, only mine 119, only Denis 90, both wrong 281
               at -4: both right 6304, only mine 341, only Denis 47, both wrong 59


## Save the picks and the shifted predictions of all fish

In [4]:
cols = {'eod_row': idx, 'frame': fr, 'fish0_head_moved_cm': mv, 'pick_denis': den}
for o in OFFS:
    cols.update({f'd{f}_{o}': D[o][:, f] for f in FISH}); cols[f'pick_{o}'] = PK[o]
cols.update({f'r_p{q_}': S0[f'r_p{q_}'].to_numpy() for q_ in PIDS})
for o in OFFS[1:]:
    cols.update({f'u0_{o}_p{q_}': S0[f'u_{o}_p{q_}'].to_numpy() for q_ in PIDS})
    cols.update({f'u{f}_{o}_p{q_}': SO[f'u{f}_{o}_p{q_}'].to_numpy() for f in (1, 2) for q_ in PIDS})
out = pd.DataFrame(cols)
path = f'{RES}/offset_all_fish_good_picks.csv'; out.to_csv(path, index=False)
print(f'saved {len(out)} EODs: d and pick of every fish at offsets {OFFS}, recorded r, model u of fish 0, 1, 2 at {OFFS[1:]} -> {path}')

saved 6751 EODs: d and pick of every fish at offsets [0, -3, -4, -5], recorded r, model u of fish 0, 1, 2 at [-3, -4, -5] -> results/eod_ground_truth_3fish/conductivities/offset_all_fish_good_picks.csv


## Figure

In [5]:
fig, ax = plt.subplots(2, 2, figsize=(15, 10))
A = ax[0, 0]; xs = np.arange(len(OFFS)); wr = 100 - T.pct_correct.to_numpy()
A.bar(xs, T.pct_fish1, color='#d98b3a', edgecolor='k', label='picked fish 1')
A.bar(xs, T.pct_fish2, bottom=T.pct_fish1, color='#7a5cc2', edgecolor='k', label='picked fish 2')
for x_, v, k_ in zip(xs, wr, T.n_correct): A.text(x_, v + 0.15, f'{v:.1f} % wrong\n({n - k_} EODs)', ha='center', va='bottom', fontsize=11)
A.axhline(100 - (den == 0).mean() * 100, color='C1', ls='--', label="Denis (EOD's own frame)")
A.set_xticks(xs, [f'{o:+d} ({o * 20:+d} ms)' if o else "0 (EOD's frame)" for o in OFFS]); A.set_ylim(0, max(wr.max(), 100 - (den == 0).mean() * 100) * 1.3)
A.set_ylabel('% of EODs not picked as fish 0'); A.set_title(f'wrong picks, {n} good-frame EODs'); A.legend(loc='upper right')

A = ax[0, 1]; ct = pd.crosstab(PK[0], PK[KB]).reindex(index=FISH, columns=FISH, fill_value=0).to_numpy()
A.imshow(np.log10(ct + 1), cmap='Greys'); A.grid(False)
for i_ in FISH:
    for j_ in FISH:
        A.text(j_, i_, ct[i_, j_], ha='center', va='center', fontsize=14, color='w' if np.log10(ct[i_, j_] + 1) > 2.2 else 'k')
A.set_xticks(FISH, [f'fish {f}' for f in FISH]); A.set_yticks(FISH, [f'fish {f}' for f in FISH])
A.set_xlabel(f'pick at {KB} frames ({KB * 20} ms)'); A.set_ylabel("pick at the EOD's frame")
A.set_title(f'picks: 0 vs {KB} (McNemar p = {binomtest(b, b + cc).pvalue:.0e})')

A = ax[1, 0]; bins = np.linspace(-2, 4, 61)
for o, col in ((0, '#888'), (KB, '#2a9d2a')):
    mg = np.nanmin(D[o][:, 1:], axis=1) - D[o][:, 0]
    A.hist(mg, bins, histtype='step', lw=2, color=col, label=f'offset {o:+d}: median {np.nanmedian(mg):.2f}, < 0 in {(mg < 0).mean() * 100:.1f} %')
A.axvline(0, color='k', lw=0.8); A.set_xlabel('margin: d of the closest other fish − d of fish 0 (> 0: fish 0 picked)')
A.set_ylabel('number of EODs'); A.set_title('how clearly fish 0 wins'); A.legend()

A = ax[1, 1]; o_ = np.argsort(fr); W = 301
for xx, col, lab in ((x0, '#888', "my pick, offset 0"), (xb, '#2a9d2a', f'my pick, offset {KB}'), (dn, 'C1', 'Denis')):
    A.plot(fr[o_], pd.Series(xx[o_].astype(float)).rolling(W, center=True, min_periods=50).mean() * 100, color=col, lw=1.8, label=lab)
A.set_xlabel('video frame of the EOD'); A.set_ylabel(f'% correct (rolling, {W} EODs)'); A.set_title('% correct over the recording'); A.legend()
fig.suptitle(f'All candidate fish solved {-KB * 20} ms earlier: who gets picked?', fontsize=15)
fig.tight_layout(); fig.savefig(f'{RES}/debug_frames/offset_all_fish_good.png', dpi=100, bbox_inches='tight'); plt.show()

/var/folders/w4/p225s7cd19n0j6tss7pdl21w0000gn/T/ipykernel_21869/3335821526.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(f'{RES}/debug_frames/offset_all_fish_good.png', dpi=100, bbox_inches='tight'); plt.show()
